<a href="https://colab.research.google.com/github/sergi-villanueva/Xatbot-1.4---Sergi-Villanueva/blob/main/XatBot_talent_2026.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from google.colab import userdata
import google.generativeai as genai
import requests
from bs4 import BeautifulSoup
from flask import Flask, request, jsonify
from flask_cors import CORS
from pyngrok import ngrok

# =================== SECRETS ===================
GEMINI_API_KEY = userdata.get('GEMINI_API_KEY')
NGROK_AUTH_TOKEN = userdata.get('NGROK_AUTH_TOKEN')

if not GEMINI_API_KEY or not NGROK_AUTH_TOKEN:
    raise ValueError("❌ Faltan Secrets. Comprova GEMINI_API_KEY i NGROK_AUTH_TOKEN")

ngrok.set_auth_token(NGROK_AUTH_TOKEN)

genai.configure(api_key=GEMINI_API_KEY)
model = genai.GenerativeModel('gemini-1.5-flash')

app = Flask(__name__)
CORS(app)

@app.route('/')
def home():
    return "✅ XatBot actiu!"

@app.route('/chat', methods=['POST'])
def chat():
    try:
        user_message = request.json.get('message', '') if request.json else ''

        WORDPRESS_URL = "https://svillanueva.inscastellbisbal.net"   # ← CANVIA AQUESTA URL

        headers = {'User-Agent': 'Mozilla/5.0'}
        r = requests.get(WORDPRESS_URL, headers=headers, timeout=10)
        soup = BeautifulSoup(r.text, 'html.parser')

        texts = [tag.get_text(strip=True) for tag in soup.find_all(['h1','h2','p','li']) if len(tag.get_text(strip=True)) > 20]
        context = " ".join(texts[:40])

        prompt = f"""Ets un assistent professional de Sergi Villanueva.
        Respon en català, clar i amable.

        Info de la web: {context}

        Pregunta: {user_message}"""

        response = model.generate_content(prompt)
        return jsonify({"reply": response.text})

    except Exception as e:
        print(f"Error: {e}")
        return jsonify({"reply": "Ho sento, hi ha hagut un error."})

# ===================== INICIAR =====================
public_url = ngrok.connect(5000)
print("\n" + "="*60)
print("✅ XATBOT EN FUNCIONAMENT")
print(f"🔗 URL: {public_url}")
print("="*60)

app.run(port=5000)

/usr/local/lib/python3.12/dist-packages/google/colab/_import_hooks/_hook_injector.py:55: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  loader.exec_module(module)


ModuleNotFoundError: No module named 'flask_cors'